# PyTorch : la carte du premier apprentissage

Situer les outils, puis construire une petite boucle avant les cas plus complexes.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#pytorch-en-une-page).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Notre but est de lire et modifier une petite boucle de classification. Nous commencerons par trois extraits fictifs, sans fichiers audio ni GPU.

Nous savons déjà programmer en Python. Nous allons apprendre les conventions particulières de PyTorch au moment où le programme en a besoin.

| Brique | Ce qu’elle fournit | Question à laquelle elle répond |
| --- | --- | --- |
| torch | Tenseurs et calculs | Quelles valeurs et quelles formes ? |
| torch.nn | Couches et pertes | Comment produire des scores et mesurer leur erreur ? |
| torch.autograd | Dérivées des opérations | Quels poids influencent la perte ? |
| torch.optim | Mise à jour des paramètres | Comment utiliser les dérivées ? |
| torch.utils.data | Exemples et chargement par lots | Comment alimenter la boucle avec davantage de données ? |

Le chargeur de données viendra après une première boucle sur un petit tableau. Nous n’avons pas besoin de tout apprendre simultanément.

La chaîne visée est simple : données → scores → perte → gradients → modification des poids. Un gradient sera expliqué par un calcul manuel, avant son utilisation dans la boucle.

## Démonstration 1 · Le programme du chapitre

Aperçu facultatif : ce tableau a trois lignes et deux colonnes. Le chapitre suivant explique sa création et sa forme.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [4]:
mesures = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]])


In [5]:
observer('mesures', mesures)

mesures forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [6]:
print(tuple(mesures.shape))

(3, 2)


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Trois extraits possèdent chacun deux descripteurs. Prévoyez la forme, avant de lancer cette cellule.

In [7]:
exo_mesures = torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(exo_mesures.shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

La première taille compte les lignes, la seconde les valeurs de chaque ligne.

</details>

### Exercice 2

Combien d’exemples ce tableau représente-t-il ? Le nombre de valeurs stockées est une autre question.

In [8]:
exo_mesures = torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_nombre = ...  # Remplacer les points par votre réponse.
if exo_nombre is Ellipsis or exo_nombre is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_nombre == exo_mesures.shape[0]
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Chaque extrait correspond à une ligne entière, pas à une case.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_mesures = torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_forme = (3, 2)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(exo_mesures.shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_mesures = torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_nombre = 3
if exo_nombre is Ellipsis or exo_nombre is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_nombre == exo_mesures.shape[0]
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [9]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()